# WOE Credit Scoring v3 - Usage Example

Este notebook demuestra el uso end-to-end de la libreria `woe_credit_scoring` (v3), que incorpora:

- Funciones EDA como `dataset_profile()`
- Validacion de configuracion con el modelo Pydantic `PipelineConfig`
- Pipeline automatizado con `AutoCreditScoring`
- Generacion de reportes con `save_reports()`
- Calculo rapido de IV con `IVCalculator`
- Interpretacion del scorecard

Los datos de ejemplo provienen de `../example_data/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from woe_credit_scoring import (
    AutoCreditScoring,
    IVCalculator,
    dataset_profile,
    PipelineConfig,
    FeatureInfo,
    ScorecardResult,
    DatasetProfile,
    frequency_table,
)
from pydantic import ValidationError

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
os.environ['PYTHONWARNINGS'] = 'ignore'

print("Librerias importadas correctamente.")

## 1. Carga de datos de ejemplo

In [ ]:
train = pd.read_csv('../example_data/train.csv')
valid = pd.read_csv('../example_data/valid.csv')

vard = [v for v in train.columns if v[:2] == 'D_']
varc = [v for v in train.columns if v[:2] == 'C_']

print(f"Train shape: {train.shape}")
print(f"Valid shape: {valid.shape}")
print(f"Discrete features ({len(vard)}): {vard[:5]}...")
print(f"Continuous features ({len(varc)}): {varc[:5]}...")
print(f"Target distribution (train):\n{train['TARGET'].value_counts(normalize=True)}")

## 2. Perfil del dataset con `dataset_profile()`

La funcion `dataset_profile()` proporciona un resumen completo del dataset: dimensiones, consumo de memoria, porcentaje de valores faltantes, distribucion del target y conteo de features por tipo.

In [ ]:
profile = dataset_profile(train, target='TARGET', discrete_features=vard, continuous_features=varc)

print("Informacion basica:")
for k, v in profile['basic_info'].items():
    print(f"  {k}: {v}")

print(f"\nTipos de features:")
for k, v in profile['feature_types'].items():
    print(f"  {k}: {v}")

print(f"\nDistribucion del target:")
print(profile['target_distribution'])

print(f"\nTop 5 features con mas missing:")
print(profile['missing_report'].sort_values('missing_pct', ascending=False).head())

### Validacion con el modelo Pydantic `DatasetProfile`

El perfil puede validarse estructuralmente usando el modelo `DatasetProfile`.

In [ ]:
missing_pct_dict = (
    profile['missing_report']
    .set_index('feature')['missing_pct']
    .to_dict()
)

profile_model = DatasetProfile(
    n_rows=profile['basic_info']['rows'],
    n_columns=profile['basic_info']['columns'],
    n_continuous=profile['feature_types']['continuous'],
    n_discrete=profile['feature_types']['discrete'],
    target_rate=float(train['TARGET'].mean()),
    missing_pct=missing_pct_dict,
)

print("DatasetProfile validado:", profile_model.model_dump_json(indent=2))

## 3. Validacion de configuracion con `PipelineConfig`

`PipelineConfig` es un modelo Pydantic v2 que valida los parametros del pipeline antes de ejecutarlo. Esto permite detectar errores de configuracion de forma temprana.

In [ ]:
config = PipelineConfig(
    iv_threshold=0.05,
    normalization_threshold=0.1,
    pdo=20,
    base_score=600,
    base_odds=50,
    min_score=400,
    max_score=900,
    discretization_method='dcc',
    max_discretization_bins=6,
    strictly_monotonic=True,
    n_threads=4,
    treat_outliers=False,
)

print("Configuracion valida:")
print(config.model_dump_json(indent=2))

### Demostracion de manejo de errores: configuracion invalida

Pydantic lanza `ValidationError` si la configuracion no cumple las restricciones. Por ejemplo, `min_score` debe ser estrictamente menor que `max_score`.

In [ ]:
try:
    PipelineConfig(min_score=800, max_score=600)
except ValidationError as e:
    print("ValidationError capturado:")
    for error in e.errors():
        print(f"  Campo: {error['loc']}, Error: {error['msg']}")

In [ ]:
try:
    PipelineConfig(iv_threshold=-0.5)
except ValidationError as e:
    print("ValidationError capturado:")
    for error in e.errors():
        print(f"  Campo: {error['loc']}, Error: {error['msg']}")

In [ ]:
try:
    PipelineConfig(max_discretization_bins=100)
except ValidationError as e:
    print("ValidationError capturado:")
    for error in e.errors():
        print(f"  Campo: {error['loc']}, Error: {error['msg']}")

## 4. Pipeline automatico con `AutoCreditScoring`

`AutoCreditScoring` ejecuta todo el pipeline: particion de datos, normalizacion de features discretas, seleccion de features basada en IV, transformacion WoE, entrenamiento del modelo de regresion logistica, scoring y reportes.

In [ ]:
acs = AutoCreditScoring(
    data=train,
    target='TARGET',
    continuous_features=varc,
    discrete_features=vard,
    random_state=42,
)

acs.fit(
    iv_feature_threshold=config.iv_threshold,
    max_discretization_bins=config.max_discretization_bins,
    strictly_monotonic=config.strictly_monotonic,
    discretization_method=config.discretization_method,
    discrete_normalization_threshold=config.normalization_threshold,
    n_threads=config.n_threads,
    treat_outliers=config.treat_outliers,
    min_score=config.min_score,
    max_score=config.max_score,
    create_reporting=True,
    verbose=True,
)

print(f"\nAUC Train: {acs.auc_train:.4f}")
print(f"AUC Valid: {acs.auc_valid:.4f}")
print(f"Features seleccionadas: {acs.candidate_features}")

### Prediccion sobre datos de validacion externa

In [ ]:
scored_valid = acs.predict(valid)
scored_valid.head(10)

### Distribucion de scores en train vs valid

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(acs.scored_train['score'], kde=False, stat='density', ax=ax, label='Train', color='steelblue', alpha=0.5)
sns.histplot(acs.scored_valid['score'], kde=False, stat='density', ax=ax, label='Valid', color='darkorange', alpha=0.5)
ax.set_title('Distribucion de Scores - Train vs Valid')
ax.set_xlabel('Score')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Generacion y guardado de reportes con `save_reports()`

In [ ]:
report_dir = '../reports/v3_demo'
acs.save_reports(report_dir)
print(f"Reportes guardados en: {os.path.abspath(report_dir)}")
for f in sorted(os.listdir(report_dir)):
    print(f"  {f}")

## 6. Calculo rapido de IV con `IVCalculator`

`IVCalculator` permite evaluar rapidamente el poder predictivo de cada feature sin ejecutar el pipeline completo.

In [ ]:
iv_calc = IVCalculator(
    data=train,
    target='TARGET',
    continuous_features=varc,
    discrete_features=vard,
)

iv_report = iv_calc.calculate_iv(
    max_discretization_bins=5,
    strictly_monotonic=False,
    discretization_method='quantile',
    discrete_normalization_threshold=0.05,
)

iv_report

### Visualizacion del reporte de IV

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
iv_sorted = iv_report.sort_values('iv')
colors = ['steelblue' if t == 'continuous' else 'darkorange' for t in iv_sorted['feature_type']]
ax.barh(iv_sorted['feature'], iv_sorted['iv'], color=colors)
ax.axvline(x=0.05, color='red', linestyle='--', label='IV = 0.05 (umbral tipico)')
ax.axvline(x=0.1, color='darkred', linestyle='--', label='IV = 0.10')
ax.set_xlabel('Information Value (IV)')
ax.set_title('Information Value por Feature')
ax.legend(
    [plt.Rectangle((0, 0), 1, 1, color='steelblue'),
     plt.Rectangle((0, 0), 1, 1, color='darkorange')],
    ['Continuous', 'Discrete']
)
plt.tight_layout()
plt.show()

## 7. Interpretacion del Scorecard

El scorecard muestra los puntos asignados a cada categoria de cada feature. A mayor puntaje, menor riesgo de default.

In [ ]:
scorecard = acs.credit_scoring.scorecard.copy()
scorecard

### Analisis de contribucion de features al score

Se puede inspeccionar como cada feature contribuye al puntaje final. Las columnas `pts_*` en la salida de `predict()` muestran los puntos de cada feature antes de sumarlos.

In [ ]:
scored_train = acs.predict(train)
pts_cols = [c for c in scored_train.columns if c.startswith('pts_')]

print("Contribucion promedio de cada feature al score (train):")
for col in sorted(pts_cols):
    print(f"  {col}: media={scored_train[col].mean():.1f}, std={scored_train[col].std():.1f}")

print(f"\nScore total (train): media={scored_train['score'].mean():.1f}, std={scored_train['score'].std():.1f}")

### Rangos de score y tasa de eventos

Los scores se agrupan en rangos para analizar la tasa de eventos (default) en cada tramo.

In [ ]:
scored_train_with_target = scored_train.copy()
scored_train_with_target['TARGET'] = train['TARGET'].values

event_rate_by_range = (
    scored_train_with_target
    .groupby('range_score_5', observed=False)['TARGET']
    .agg(['count', 'mean'])
    .rename(columns={'mean': 'event_rate'})
    .reset_index()
)
event_rate_by_range['event_rate'] = (event_rate_by_range['event_rate'] * 100).round(2)

print("Tasa de eventos por rango de score (5 bins):")
event_rate_by_range

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

bars = ax1.bar(
    event_rate_by_range['range_score_5'].astype(str),
    event_rate_by_range['count'],
    color='steelblue', alpha=0.6, label='Cantidad'
)
ax1.set_xlabel('Rango de Score')
ax1.set_ylabel('Cantidad de casos', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

ax2 = ax1.twinx()
ax2.plot(
    event_rate_by_range['range_score_5'].astype(str),
    event_rate_by_range['event_rate'],
    color='darkred', marker='o', linewidth=2, label='Tasa de evento (%)'
)
ax2.set_ylabel('Tasa de evento (%)', color='darkred')
ax2.tick_params(axis='y', labelcolor='darkred')

ax1.set_title('Distribucion de scores y tasa de eventos por rango')
fig.tight_layout()
plt.show()

## Resumen

Este notebook demostro el flujo completo de la libreria `woe_credit_scoring` v3:

1. **EDA**: `dataset_profile()` para un perfil rapido del dataset.
2. **Validacion**: `PipelineConfig` (Pydantic) para validar parametros antes de ejecutar.
3. **Pipeline**: `AutoCreditScoring` con normalizacion, seleccion IV, WoE, regresion logistica y scoring.
4. **Reportes**: `save_reports()` para exportar graficos a PNG.
5. **IV rapido**: `IVCalculator` para evaluar poder predictivo de features.
6. **Interpretacion**: Scorecard con puntos por categoria y analisis de contribucion.